In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import ot
from scanpy import AnnData
from tqdm import tqdm
import scipy
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import time

import warnings
warnings.filterwarnings("ignore")

/slurm/home/yrd/fanlab/qianjingyang/.conda/envs/sccube/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

In [3]:
def pre_filter(
        adata_st: AnnData,
        adata_sc: AnnData,

):
    # if issparse(adata_sc.X):
    #     adata_sc.X = adata_sc.X.toarray()
    # if issparse(adata_st.X):
    #     adata_st.X = adata_st.X.toarray()
        
    adata_sc.var_names_make_unique()
    adata_st.var_names_make_unique()

    # filter genes, cells, and spots
    # print(f"Filter data")
    # sc.pp.filter_cells(adata_sc, min_genes=1)
    # sc.pp.filter_genes(adata_sc, min_cells=1)
    # sc.pp.filter_cells(adata_st, min_genes=1)
    # sc.pp.filter_genes(adata_st, min_cells=1)
    # print(f"Done")
    
    return adata_sc, adata_st


def estimate_cell_number(
        adata,
        mean_cell_numbers,
        normalize: bool = True,
):
    """

    :param adata_st:
    :param mean_cell_numbers:
    :return:
    """
    assert mean_cell_numbers > 0, 'Mean cell number must be positive!'

    adata_use = adata.copy()
    if normalize:
        print('Estimating based on normalized data')
        sc.pp.normalize_total(adata_use, target_sum=1e6)
        sc.pp.log1p(adata_use)
        
    if issparse(adata_use.X):
        adata_use.X = adata_use.X.toarray()
    
    expr = adata_use.X.T.astype(float)

    # Set up fitting problem
    RNA_reads = np.sum(expr, axis=0, dtype=float)
    mean_RNA_reads = np.mean(RNA_reads)
    min_RNA_reads = np.min(RNA_reads)

    min_cell_numbers = 1 if min_RNA_reads > 0 else 0

    fit_parameters = np.polyfit(np.array([min_RNA_reads, mean_RNA_reads]),
                                np.array([min_cell_numbers, mean_cell_numbers]), 1)
    polynomial = np.poly1d(fit_parameters)
    estimated_cell_number = np.round(polynomial(RNA_reads)).astype(int)
    adata.obs['estimated_cell_number'] = estimated_cell_number
    print(f"Done")
    
    return adata


def integer_allocation(prop, counts):
    expected_cells = prop * counts[:, None]
    int_cells = np.floor(expected_cells).astype(int)
    frac_cells = expected_cells - int_cells
    remaining_cells = counts - int_cells.sum(axis=1)
    for i in range(len(counts)):
        frac_order = np.argsort(-frac_cells[i])
        for j in range(remaining_cells[i]):
            int_cells[i, frac_order[j]] += 1
    return int_cells


def adjust_abundance(
    adata_st: AnnData,
    adata_sc: AnnData,
    celltype_key: str = 'celltype',
):
    celltype_unique = sorted(set(adata_sc.obs[celltype_key]))
    
    cell_counts = np.array(adata_st.obs['estimated_cell_number'])
    prop = np.array(adata_st.obs[celltype_unique].copy())
    map_target = integer_allocation(prop, cell_counts)
    
    target_num_list = map_target.sum(axis=0)
    sc_num_list = np.array(adata_sc.obs[celltype_key].value_counts()[celltype_unique])
    diff_num_list = sc_num_list - target_num_list
    
    adata_list = []
    for i in range(len(celltype_unique)):
        adata_tmp = adata_sc[adata_sc.obs[celltype_key] == celltype_unique[i]].copy()
        adata_list.append(adata_tmp)
        
    print(f"Adjust abundance of each cell types")
    for i in tqdm(range(len(celltype_unique))):
        adjust_num = diff_num_list[i]
        adata_tmp = adata_list[i].copy()
        if adjust_num >= 0:
            set_seed(0)
            selected_indices = np.random.choice(adata_tmp.shape[0], size=target_num_list[i], replace=False)
            adata_tmp = adata_tmp[selected_indices]
        elif adjust_num < 0:
            fold = np.abs(diff_num_list[i]) / sc_num_list[i]
            if fold > 1:
                fold_int = int(fold)
                selected_indices = list(range(adata_tmp.shape[0])) * fold_int
                set_seed(0)
                selected_indices2 = list(
                    np.random.choice(adata_tmp.shape[0], size=(np.abs(diff_num_list[i]) - fold_int * adata_tmp.shape[0]), replace=False)
                )
                selected_indices.extend(selected_indices2)
                selected_indices = np.array(selected_indices)
            else:
                set_seed(0)
                selected_indices = np.random.choice(adata_tmp.shape[0], size=np.abs(diff_num_list[i]), replace=False)
            
            adata_tmp_replicate = adata_tmp[selected_indices].copy()
            adata_tmp = sc.concat([adata_tmp, adata_tmp_replicate]).copy()
            
        adata_list[i] = adata_tmp.copy()
        
    adata_sc_new = sc.concat(adata_list).copy()
    adata_sc_new.obs_names_make_unique()
    print(f"Done")
    
    return adata_st, adata_sc_new


def preprocess(
    adata_st: AnnData,
    adata_sc: AnnData,
    scale: bool = False,
):
    # copy
    adata_st_raw = adata_st.copy()
    adata_sc_raw = adata_sc.copy()
    
    print(f"Normalize data")
    sc.pp.normalize_total(adata_st, target_sum=1e6)
    sc.pp.log1p(adata_st)
    sc.pp.normalize_total(adata_sc, target_sum=1e6)
    sc.pp.log1p(adata_sc)
    
    if scale:
        print(f"Scale data")
        sc.pp.scale(adata_st)
        sc.pp.scale(adata_sc)
        
    print(f"Done")
    
    return adata_st_raw, adata_sc_raw, adata_st, adata_sc


from sklearn.metrics.pairwise import cosine_similarity

def OT(
    adata_st: AnnData,
    adata_sc: AnnData,
    numItermax: int = 1e6
):
    Xs = adata_sc.X.copy()
    Xt = adata_st.X.copy()
    
    if issparse(Xs):
        Xs = Xs.toarray()
    if issparse(Xt):
        Xt = Xt.toarray()
    
    print(f"Calculate cost matrix (cosine similarity)")
    cosine_sim_matrix = cosine_similarity(Xs, Xt)
    M = 1 - cosine_sim_matrix
    M /= M.max()
    
    # weights
    a = np.ones((adata_sc.shape[0],))
    b = np.array(adata_st.obs['estimated_cell_number'])
    
    print(f"Run OT")
    transport_matrix = ot.emd(a, b, M, numItermax=numItermax)
    print(f"Done")
    
    return transport_matrix


from sklearn.neighbors import NearestNeighbors

def jitter_coord(coord):
    # cell number
    num = coord.shape[0]
    # min distance
    coord_unique = np.unique(coord, axis=0)
    nbrs = NearestNeighbors(n_neighbors=2).fit(coord_unique)
    distances, indices = nbrs.kneighbors(coord_unique)
    min_distance = min(distances[:, -1][distances[:, -1] > 0])

    x_list = list(coord[:, 0])
    y_list = list(coord[:, 1])

    set_seed(0)
    length = np.random.uniform(0, min_distance, num)
    radius = np.pi * np.random.uniform(0, 2, num)

    x_list_new = x_list + length * np.cos(radius)
    y_list_new = y_list + length * np.sin(radius)
    coord_new = np.array([[x_list_new[i], y_list_new[i]] for i in range(num)])

    return coord_new


def process_result(
    adata_st: AnnData,
    adata_sc: AnnData,
    transport_matrix: np.array,
    celltype_key: str = 'celltype',
):
    
    print(f"Assign cells")
    spot_to_cells = []
    cell_counts = adata_st.obs['estimated_cell_number']
    for i in tqdm(range(transport_matrix.shape[1])):
        k = int(cell_counts[i])
        top_k_cells = np.argsort(-transport_matrix[:, i])[:k]
        spot_to_cells.append(list(top_k_cells))
        
    print(f"Create new data")
    if issparse(adata_sc.X):
        adata_sc.X = adata_sc.X.toarray()
        
    # original
    original_spot = list(adata_st.obs_names)
    original_cell = list(adata_sc.obs_names)
    original_celltype = list(adata_sc.obs[celltype_key])
    original_x = list(adata_st.obsm['spatial'][:, 0])
    original_y = list(adata_st.obsm['spatial'][:, 1])
    original_expr = adata_sc.X
    
    # new
    cell_list = []
    celltype_list = []
    spot_list = []
    x_list = []
    y_list = []
    expr_list = []
    
    for i, indices in enumerate(spot_to_cells):
        cell_list.extend(original_cell[idx] for idx in indices)
        celltype_list.extend(original_celltype[idx] for idx in indices)

        spot_list.extend([original_spot[i]] * len(indices))
        x_list.extend([original_x[i]] * len(indices))
        y_list.extend([original_y[i]] * len(indices))

        expr_list.extend(original_expr[indices])
        
    new_id_list = ['CID' + str(i + 1) for i in range(len(cell_list))]
    
    new_meta = pd.DataFrame({
        'NewCID': new_id_list,
        'OriginalCID': cell_list,
        'CellType': celltype_list,
        'SpotID': spot_list,
        'X': x_list,
        'Y': y_list,
    })
    
    new_meta.index = new_id_list
    new_expr = np.array(expr_list)
    new_expr = scipy.sparse.csr_matrix(new_expr)
    coord = np.array(new_meta[['X', 'Y']])
    coord_jitter = jitter_coord(coord)
    
    new_meta['X_jitter'] = coord_jitter[:, 0]
    new_meta['Y_jitter'] = coord_jitter[:, 1]

    # new AnnData
    adata_new = sc.AnnData(new_expr)
    adata_new.obs = new_meta
    adata_new.obs_names = new_id_list
    adata_new.var_names = adata_sc_raw.var_names
    adata_new.obsm['spatial'] = coord_jitter
    
    print("Done")
    return adata_new

In [4]:
noise_list = ['0', '05', '10', '20', '40']
n_list = [5, 10, 15]

for noise in noise_list:
    for n in n_list:
        sc_adata = sc.read('../output/Cerebellum_sc_noise' + noise + '.h5ad')
        st_adata = sc.read('../output/Cerebellum_st_n' + str(n) + '.h5ad')
        
        # spot_x, spot_y, spot
        st_adata.obs = st_adata.obs.iloc[:, -3:]
        
        sc_adata, st_adata = pre_filter(
            adata_st=st_adata.copy(),
            adata_sc=sc_adata.copy(),
        )
        
        st_adata = estimate_cell_number(
            adata=st_adata.copy(),
            mean_cell_numbers=n,
            normalize=True,
        )
        
        # load deconvolution results
        prop = pd.read_csv('../results/C2L_Noise' + noise + '_n' + str(n) + '.csv', index_col=0)
        ct_sort = sorted(set(sc_adata.obs['CellType']))
        prop = prop[ct_sort]
        prop = prop.loc[st_adata.obs_names]
        obs_raw = st_adata.obs.copy()
        obs_new = pd.concat([obs_raw, prop], axis=1)
        st_adata.obs = obs_new
        
        st_adata, sc_adata = adjust_abundance(
            adata_st = st_adata.copy(),
            adata_sc = sc_adata.copy(),
            celltype_key='CellType',
        )
        
        adata_st_raw, adata_sc_raw, adata_st, adata_sc = preprocess(
            adata_st = st_adata.copy(),
            adata_sc = sc_adata.copy(),
            scale=False,
        ) 
        
        transport_matrix = OT(
            adata_st=adata_st.copy(),
            adata_sc=adata_sc.copy(),
            numItermax=1e6
        )
        
        adata_new = process_result(
            adata_st=adata_st_raw.copy(),
            adata_sc=adata_sc_raw.copy(),
            transport_matrix=transport_matrix,
            celltype_key='CellType',
        )
        
        adata_new.write('../results/Cerebellum_scPositioner_noise' + noise + '_n' + str(n) + '.h5ad')

Estimating based on normalized data
Done
Adjust abundance of each cell types


100%|██████████| 11/11 [00:01<00:00,  5.83it/s]


Done
Normalize data
Done
Calculate cost matrix (cosine similarity)
Run OT
Done
Assign cells


100%|██████████| 4496/4496 [00:01<00:00, 2433.11it/s]


Create new data
Done
Estimating based on normalized data
Done
Adjust abundance of each cell types


...

Create new data
Done
Estimating based on normalized data
Done
Adjust abundance of each cell types


100%|██████████| 11/11 [00:01<00:00, 10.60it/s]


Done
Normalize data
Done
Calculate cost matrix (cosine similarity)
Run OT
Done
Assign cells


100%|██████████| 1564/1564 [00:00<00:00, 2311.48it/s]


Create new data
Done
